# Single-channel beam fit at 156.25 MHz

**Question.** The empirical PCA beam (v0003, $\ell_{\max}=8$) misses smooth structure in the 156.25 MHz map. Does fitting this channel alone, with no bridging across frequency, capture that structure?

**Model.** $d_i \simeq \left|\sum_{\ell m}\mathbf c_{\ell m}\,Y_{\ell m}(\Omega_i)\cdot\mathbf e_i\right|^2$ with $\ell_{\max}=8$. The $\mathbf c_{\ell m}$ are complex Cartesian 3-vectors on real spherical harmonics, the same basis as the DPSS/PCA fits. $\mathbf e_i$ is the arm-0 transmitter vector projected transverse to the line of sight. There is no HFSS prior and no spectral basis. With the coefficients free, a separate amplitude is degenerate with their overall scale. The fit therefore has no separate amplitude parameter. With one transmitter arm, only the component of the field along $\mathbf e_i$ is measured; the orthogonal transverse component is unconstrained, and so is the beam's total normalization (see the polarization table below). The overall phase is unobservable too.

**Fixed inputs**, identical to v0003 so that only the frequency bridging (and its HFSS prior) is removed:
- The 07-17 20:26:00–21:28:40 UTC raster (phase C, box-air, RFANT) and the tooth − gap signal.
- Pointing for the main fit from `marjum-2026-07/point_table_v1-beta.npz`. A later section compares it with commanded azimuth, which fits far better.
- `point_table_v1-beta.npz` is `pointing_table@v1.2` with az held constant through each el sweep, because the az and el drives never run together. It is generated by `data-analysis/scripts/marjum-2026-07/build_point_table_v1_beta.py`. As before, 89 reversal glitches are dropped, the dropout mask is applied, and the 56-sample opening el≈0 slew is excluded.
- The v0003 PCA comparison beam was fitted with `pointing_table@v1.2` az. So its fit, and the HFSS reference taken from v0003, use slightly different pointing from the single-channel fit here.
- v0003 geometry: $\Delta\alpha=\Delta az=\Delta el=0$, transmitter offset $(-1.06, +0.42)$ m E/N.
- The 12° azimuth-stripe train/validation/test split.

The fit starts from the HFSS 156.25 MHz field projected onto the harmonics and scaled to the data. The loss is unweighted least squares in counts, as in v0003. A training-stripe fit gives the held-out scores, and an all-sample fit gives the maps.

**Reading the result.** If the single-channel fit captures the smooth structure that v0003 misses, frequency bridging (or its prior) is implicated. If it doesn't, the cause lies in $\ell_{\max}=8$ itself, the weighting, or the forward model.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import healpy as hp
import pyarrow.parquet as pq

PLOT_BACKEND = os.environ.get('EIGSEP_NOTEBOOK_BACKEND', 'widget')
get_ipython().run_line_magic('matplotlib', PLOT_BACKEND)
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
os.environ.setdefault('JAX_ENABLE_X64', 'true')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'marjum-2026-07/data').is_dir())
CAMPAIGN = ROOT / 'marjum-2026-07'

import eigsep_data
from eigsep_data import AntennaResolutionPolicy, MetadataIndex
from eigsep_data.beam_sim import read_beam
from scipy.special import sph_harm_y
from scipy.optimize import minimize
from IPython.display import display
import hashlib
import jax
import jax.numpy as jnp
jax.config.update('jax_enable_x64', True)
eigsep_data.set_campaign_root(CAMPAIGN)

RASTER_UTC = ('2026-07-17 20:26:00', '2026-07-17 21:28:40')
FREQ_MHZ = 156.25            # an HFSS slice; every HFSS slice sits on an arm-1 comb tooth
GAP_OFFSET = 4

# Geometry and polarisation from beam_explorer.ipynb (branch B, roll mount).
AZ_SIGN, EL_SIGN = -1.0, -1.0
AZ0, EL0 = 177.0, 0.0        # deg
ALPHA = 45.0                 # transmitter polarisation, deg ccw from East

In [ ]:
t0, t1 = (pd.Timestamp(s, tz='UTC').timestamp() for s in RASTER_UTC)

# --- corrected pointing, joined to the correlator rows on time -------------
# point_table_v1-beta: pointing_table@v1.2 with az held constant through each el
# sweep (the az and el drives never run together), over the beam-scan raster.
with np.load(CAMPAIGN / 'point_table_v1-beta.npz') as z:
    POINT_PROVENANCE = json.loads(str(z['provenance']))
    pointing = pd.DataFrame({key: z[key] for key in ('t_utc_s', 'az_deg', 'el_deg', 'quality')})
assert pointing.t_utc_s.min() <= t0   # the table covers the raster start
pointing = pointing[(pointing.t_utc_s >= t0 - 60) & (pointing.t_utc_s <= t1 + 60)]
print('pointing: %s (%s), from %s' % (POINT_PROVENANCE['product'], POINT_PROVENANCE['version'],
                                      POINT_PROVENANCE['source_compact']))
policy = AntennaResolutionPolicy.load(CAMPAIGN / 'curation/antenna_resolution.json')
bundle = MetadataIndex(CAMPAIGN / 'data').select(time=(t0, t1)).load_bundle(
    antenna='box-air', missing='skip', resolution_policy=policy)
meta = bundle.meta.reset_index(drop=True).assign(t=lambda x: x.time_best.astype(float))
j = pd.merge_asof(meta.sort_values('t').reset_index(names='bundle_row'),
                  pointing.sort_values('t_utc_s')[['t_utc_s', 'az_deg', 'el_deg', 'quality']],
                  left_on='t', right_on='t_utc_s', direction='nearest', tolerance=0.35)
j = j[(j.quality == 'ok') & (j.rfswitch == 'RFANT')].reset_index(drop=True)
assert len(j) == 6990, len(j)   # same rows as with pointing_table@v1.2 (6901 after glitch removal)

def reversal_glitches(el_deg, tol=15.0):
    """Isolated solved-elevation faults: a sample is good if at least 3 of its
    4 neighbours (+/-1, +/-2) lie within `tol` deg of it across the +/-180
    wrap. The sweeps move ~3 deg per sample, so real data always passes.
    At reversals the IMU readout drops single samples to ~0 or -60 deg while
    the motor is pegged at -180, and those have at most one close neighbour.
    """
    el = np.asarray(el_deg, float)
    close = np.zeros(el.size, int)
    for k in (-2, -1, 1, 2):
        d = np.abs((el - np.roll(el, k) + 180.0) % 360.0 - 180.0)
        edge = slice(0, -k) if k < 0 else slice(-k, None)   # neighbours that wrapped around the array
        d[edge] = 0.0
        close += d <= tol
    return close < 3


glitch = reversal_glitches(j.el_deg.to_numpy())
print('dropped %d elevation glitches' % glitch.sum())
j = j[~glitch].reset_index(drop=True)

T = j.t.to_numpy()
AZ = j.az_deg.to_numpy()
EL = j.el_deg.to_numpy()

# --- the channel -------------------------------------------------------------
BEAM_CART, _, HFSS_FREQS = read_beam()
s = int(np.argmin(np.abs(HFSS_FREQS - FREQ_MHZ)))
ch = int(round(HFSS_FREQS[s] / (250.0 / 1024)))
col = {int(round(f / (250.0 / 1024))): i for i, f in enumerate(bundle.freqs_mhz)}
rows = j.bundle_row.to_numpy()
tooth = bundle.data[rows, col[ch]].astype(float)
gap = 0.5 * (bundle.data[rows, col[ch - GAP_OFFSET]] + bundle.data[rows, col[ch + GAP_OFFSET]])
DATA = tooth - gap
# Whole-band dropouts (e.g. 21:18:00-21:18:35) take the gap down with the tooth.
ok = (gap / np.median(gap) > 1 / 1.5) & (gap / np.median(gap) < 1.5) & np.isfinite(DATA)
print('channel %d = %.4f MHz; %d samples, %d masked as dropouts'
      % (ch, HFSS_FREQS[s], len(DATA), (~ok).sum()))

In [ ]:
# --- v0003 comparison models, masks, split and geometry -----------------------
PRODUCT = CAMPAIGN / 'derived/beam/empirical_raster_v0003'
provenance = json.loads((PRODUCT / 'provenance.json').read_text())
for relative_path, expected in provenance['artifact_sha256'].items():
    assert hashlib.sha256((PRODUCT / relative_path).read_bytes()).hexdigest() == expected, relative_path
with np.load(PRODUCT / 'pca/diagnostics.npz') as z:
    pca = {key: z[key] for key in z.files}
assert np.allclose(T, pca['t'], rtol=0, atol=1e-5)
fidx = int(np.flatnonzero(pca['channels'] == ch)[0])
ARM = int(pca['arms'][fidx])
FIT_OK = pca['good'][fidx].astype(bool)            # dropout mask and el~0 slew removed
assert np.array_equal(FIT_OK, ok & ~pca['el0_slew'].astype(bool))
SPLIT = pca['split']                                # 0 train, 1 validation, 2 test
HFSS_V0003 = pca['hfss'][fidx]                      # HFSS, training-stripe gain, v0003 geometry
PCA_V0003 = pca['empirical_exported'][fidx]         # all-sample PCA lmax=8 fit
PCA_V0003_HELDOUT = pca['empirical_heldout'][fidx]
params = pca['params']
print('channel %d (%.3f MHz), arm %d; geometry %s' % (ch, HFSS_FREQS[s], ARM, dict(zip(
    ['dalpha_deg', 'daz_deg', 'del_deg', 'tx_east_m', 'tx_north_m'], params.round(3)))))
print('samples: %d fit, split counts %s' % (FIT_OK.sum(), np.bincount(SPLIT[FIT_OK], minlength=3)))

In [ ]:
# --- forward model -------------------------------------------------------------
LMAX = 8
ant = np.asarray(json.loads((CAMPAIGN / 'curation/horizon_profiles.json').read_text())['antenna_enu_m'])
tx = np.asarray(json.loads((CAMPAIGN / 'curation/transmitter_position.json').read_text())['best_estimate_enu_m'])


def rotation(az_deg, el_deg):
    # topo = Rx(E) @ Rz(A) @ body: azimuth rolls about the boresight, elevation tips.
    A, E = np.deg2rad(az_deg), np.deg2rad(el_deg)
    cA, sA, cE, sE = np.cos(A), np.sin(A), np.cos(E), np.sin(E)
    z, o = np.zeros_like(cA), np.ones_like(cA)
    Rz = np.stack([np.stack([cA, -sA, z], -1), np.stack([sA, cA, z], -1), np.stack([z, z, o], -1)], -2)
    Rx = np.stack([np.stack([o, z, z], -1), np.stack([z, cE, -sE], -1), np.stack([z, sE, cE], -1)], -2)
    return Rx @ Rz


def harmonics(theta, phi, lmax):
    # Real orthonormal spherical harmonics, ordered as in the DPSS/PCA fits.
    columns = []
    for ell in range(lmax + 1):
        for m in range(-ell, ell + 1):
            y = sph_harm_y(ell, abs(m), theta, phi)
            columns.append(y.real if m == 0 else np.sqrt(2) * (y.imag if m < 0 else y.real))
    return np.column_stack(columns)


pa, paz, pel, peast, pnorth = params
direction = tx + np.array([peast, pnorth, 0.]) - ant
direction /= np.linalg.norm(direction)
rot = rotation(-AZ + AZ0 + paz, -EL + EL0 + pel)
body = np.einsum('nji,j->ni', rot, direction)
theta = np.arccos(np.clip(body[:, 2], -1, 1))
phi = np.mod(np.arctan2(body[:, 1], body[:, 0]), 2 * np.pi)
alpha = np.deg2rad(45 + pa + 90 * ARM)
e_tx = np.einsum('nji,j->ni', rot, [-np.sin(alpha), np.cos(alpha), 0.])
e_tx -= np.sum(e_tx * body, axis=1)[:, None] * body
Y = harmonics(theta, phi, LMAX)
DESIGN = np.concatenate([Y * e_tx[:, c, None] for c in range(3)], axis=1)   # (N, 3 nlm)

# HFSS 156.25 MHz field projected onto the same harmonics: the starting point.
NSIDE = hp.npix2nside(BEAM_CART.shape[-1])
pix_theta, pix_phi = hp.pix2ang(NSIDE, np.arange(BEAM_CART.shape[-1]))
PIX_DIR = np.array(hp.pix2vec(NSIDE, np.arange(BEAM_CART.shape[-1])))
Y_PIX = harmonics(pix_theta, pix_phi, LMAX)
C_HFSS = np.linalg.lstsq(Y_PIX, BEAM_CART[s].T, rcond=None)[0].T.ravel()    # component-blocked
print('HFSS field truncation error at lmax=%d: %.2f%%' % (LMAX, 100 * np.linalg.norm(
    BEAM_CART[s] - C_HFSS.reshape(3, -1) @ Y_PIX.T) / np.linalg.norm(BEAM_CART[s])))


def unit_power_gain(coeff):
    # Mean spherical transverse power of the field: the gain on the unit-average-power beam.
    field = coeff.reshape(3, -1) @ Y_PIX.T
    field -= np.sum(field * PIX_DIR, axis=0) * PIX_DIR
    return float(np.mean(np.sum(abs(field) ** 2, axis=0)))

In [ ]:
# --- fit: harmonic coefficients only, unweighted least squares in counts -----
def fit_channel(use, maxiter=5000, design=None):
    design = DESIGN if design is None else design
    p0 = abs(design @ C_HFSS) ** 2
    c0 = C_HFSS * np.sqrt(np.sum(p0[use] * DATA[use]) / np.sum(p0[use] ** 2))
    scale = np.sqrt(np.mean(DATA[use] ** 2))
    D, target = jnp.asarray(design[use]), jnp.asarray(DATA[use] / scale)
    n = c0.size

    def objective(x):
        return jnp.mean((jnp.abs(D @ (x[:n] + 1j * x[n:])) ** 2 - target) ** 2)

    vg = jax.jit(jax.value_and_grad(objective))
    x0 = np.r_[c0.real, c0.imag] / np.sqrt(scale)
    opt = minimize(lambda x: tuple(np.asarray(v, float) for v in vg(jnp.asarray(x))),
                   x0, jac=True, method='L-BFGS-B',
                   options=dict(maxiter=maxiter, ftol=1e-12, gtol=1e-8, maxcor=30))
    coeff = (opt.x[:n] + 1j * opt.x[n:]) * np.sqrt(scale)
    return abs(design @ coeff) ** 2, coeff, opt


def frac_rms(model, use):
    return np.linalg.norm((DATA - model)[use]) / np.linalg.norm(DATA[use])


TRAIN = FIT_OK & (SPLIT == 0)
SINGLE_HELDOUT, _, train_opt = fit_channel(TRAIN)
SINGLE, single_coeff, all_opt = fit_channel(FIT_OK)
print('real parameters %d; train fit: %d iterations, %s; all-sample fit: %d iterations, %s'
      % (2 * single_coeff.size, train_opt.nit, train_opt.message, all_opt.nit, all_opt.message))

rows = []
for name, model, held in [('HFSS (one gain)', HFSS_V0003, HFSS_V0003),
                          ('v0003 PCA lmax=8, 42 channels', PCA_V0003, PCA_V0003_HELDOUT),
                          ('single-channel lmax=8', SINGLE, SINGLE_HELDOUT)]:
    rows.append(dict(model=name, train=frac_rms(held, TRAIN),
                     validation_heldout=frac_rms(held, FIT_OK & (SPLIT == 1)),
                     test_heldout=frac_rms(held, FIT_OK & (SPLIT == 2)),
                     all_samples=frac_rms(model, FIT_OK)))
display(pd.DataFrame(rows).set_index('model').round(4))

# One transmitter arm constrains only E.e. The transverse component orthogonal to
# e never reaches the data, so an unregularized fit lets it drift; the total beam
# normalization (and so a separate amplitude) is then not identified.
def pol_power(coeff):
    E = (coeff.reshape(3, -1) @ Y.T).T[FIT_OK]
    b = body[FIT_OK]
    E -= np.sum(E * b, axis=1)[:, None] * b
    e_hat = e_tx[FIT_OK] / np.linalg.norm(e_tx[FIT_OK], axis=1)[:, None]
    return (np.mean(abs(np.sum(E * e_hat, axis=1)) ** 2),
            np.mean(abs(np.sum(E * np.cross(b, e_hat), axis=1)) ** 2))


p0 = abs(DESIGN @ C_HFSS) ** 2
start = C_HFSS * np.sqrt(np.sum(p0[FIT_OK] * DATA[FIT_OK]) / np.sum(p0[FIT_OK] ** 2))
display(pd.DataFrame([dict(field=name, co_pol=co, cross_pol=cross, cross_over_co=cross / co)
                      for name, (co, cross) in [('HFSS start', pol_power(start)),
                                                ('single-channel fit', pol_power(single_coeff))]]
                     ).set_index('field').round(3))

In [ ]:
plt.rcParams.update({'font.size': 14, 'axes.spines.top': False, 'axes.spines.right': False})
az_plot = (AZ + 180.0) % 360.0 - 180.0          # the raster crosses 0/360; keep it contiguous
time = pd.to_datetime(T, unit='s', utc=True)

# Shared color scales for both models: log data/model, symmetric residual set by HFSS.
lo, hi = np.percentile(DATA[FIT_OK & (DATA > 0)], [1, 99.5])
norm = LogNorm(vmin=lo, vmax=hi)
rlim = np.percentile(np.abs((DATA - HFSS_V0003)[FIT_OK]), 98)


def plot_maps(model, name, az=None):
    x = az_plot if az is None else (az + 180.0) % 360.0 - 180.0
    fig, axes = plt.subplots(1, 3, figsize=(19, 7), sharex=True, sharey=True,
                             constrained_layout=True)
    panels = [('Data (tooth − gap)', DATA, dict(cmap='viridis', norm=norm)),
              (name + ' model', model, dict(cmap='viridis', norm=norm)),
              ('Residual (data − model)', DATA - model, dict(cmap='RdBu_r', vmin=-rlim, vmax=rlim))]
    for ax, (title, value, kw) in zip(axes, panels):
        sc = ax.scatter(x[FIT_OK], EL[FIT_OK], c=value[FIT_OK], s=4, **kw)
        ax.set_title(title)
        ax.set_xlabel('Azimuth [deg]')
        fig.colorbar(sc, ax=ax, orientation='horizontal', pad=0.1, shrink=0.85, label='Counts')
    axes[0].set_ylabel('Elevation [deg]')
    fig.suptitle('%s, %.3f MHz, raster %s–%s UTC'
                 % (name, HFSS_FREQS[s], RASTER_UTC[0][-8:], RASTER_UTC[1][-8:]), fontsize=17)
    plt.show()


def plot_time(model, name):
    fig, ax = plt.subplots(figsize=(16, 6.5), constrained_layout=True)
    ax.plot(time[FIT_OK], DATA[FIT_OK], '.', ms=2, color='0.25', label='data')
    ax.plot(time[FIT_OK], model[FIT_OK], '.', ms=2, color='tab:red', label=name + ' model')
    ax.plot(time[FIT_OK], (DATA - model)[FIT_OK], '.', ms=2, color='tab:blue', label='residual')
    ax.axhline(0, color='0.6', lw=1)
    ax.set_ylabel('Counts')
    ax.set_xlabel('Time [UTC]')
    ax.set_title('%s, %.3f MHz through the raster' % (name, HFSS_FREQS[s]))
    ax.legend(loc='upper right', markerscale=6)
    plt.show()

## HFSS (reference)

All models use v0003 geometry and are shown on the same samples and color scales.

In [ ]:
plot_maps(HFSS_V0003, 'HFSS')
plot_time(HFSS_V0003, 'HFSS')

## v0003 PCA, $\ell_{\max}=8$, bridged across 42 channels

In [ ]:
plot_maps(PCA_V0003, 'v0003 PCA ℓmax=8')
plot_time(PCA_V0003, 'v0003 PCA ℓmax=8')

## Single-channel fit, $\ell_{\max}=8$

In [ ]:
plot_maps(SINGLE, 'single-channel ℓmax=8')
plot_time(SINGLE, 'single-channel ℓmax=8')

## Commanded azimuth: did the potentiometer slip instead of the platform?

`pointing_table` puts a −27.35° azimuth slip over 20:41:24–20:53:43: the platform falls behind the motor and stays about 27.5° behind. If the potentiometer slipped instead, the platform followed the motor, and the commanded steps are the true pointing.

**Commanded azimuth** is each sample's `motor_az_deg` plus one constant offset. Motor az is exactly constant through each el sweep and steps 5.0018° between sweeps. Azimuth zero is not tied to anything absolute, and the geometry's zero (AZ0 = 177°) was tuned against pot azimuth. So the offset is chosen by the unfitted HFSS beam (one gain) on the training stripes, on a 0.5° grid; the pot pointing gets the same scan for fairness. Elevation stays the gravity-referenced IMU value. `motor_el_deg` scatters by 15° against it, and its median offset drifts from −1° to −8° over the raster, so commanded el loops are not usable.

Everything else is unchanged: samples, masks, splits, geometry, starting point and fit. The HFSS rows test the pointing without any beam fitting. Commanded-az maps are plotted on commanded azimuth.

A model-free check compares consecutive el sweeps directly: the fractional change between their elevation profiles. If the platform really stalled where the pot says so, those consecutive sweeps should be nearly identical.

In [ ]:
wrap = lambda deg: (np.asarray(deg) + 180.0) % 360.0 - 180.0
with np.load(CAMPAIGN / 'point_table_v1-beta.npz') as z:
    cmd = pd.DataFrame({key: z[key] for key in ('t_utc_s', 'motor_az_deg', 'az_deg_v1', 'sweep')})
cmd = pd.merge_asof(pd.DataFrame({'t': T}), cmd.sort_values('t_utc_s'), left_on='t', right_on='t_utc_s',
                    direction='nearest', tolerance=0.35)
assert cmd.motor_az_deg.notna().all()
MOTOR_AZ, SWEEP = cmd.motor_az_deg.to_numpy(), cmd.sweep.to_numpy()
tt = pd.to_datetime(T, unit='s', utc=True)
def window(a, b):
    return (tt >= pd.Timestamp('2026-07-17 ' + a, tz='UTC')) & (tt < pd.Timestamp('2026-07-17 ' + b, tz='UTC'))


def pointing_model(az_deg, el_deg):
    # Same forward model as above for an alternative pointing: the harmonic design
    # matrix and the unfitted HFSS power with one training-stripe gain.
    r = rotation(-az_deg + AZ0 + paz, -el_deg + EL0 + pel)
    b = np.einsum('nji,j->ni', r, direction)
    th, ph = np.arccos(np.clip(b[:, 2], -1, 1)), np.mod(np.arctan2(b[:, 1], b[:, 0]), 2 * np.pi)
    e = np.einsum('nji,j->ni', r, [-np.sin(alpha), np.cos(alpha), 0.])
    e -= np.sum(e * b, axis=1)[:, None] * b
    pix, weight = hp.get_interp_weights(NSIDE, th, ph)
    w = sum(BEAM_CART[s][:, pix[k]] * weight[k] for k in range(4)).T
    hfss = abs(np.sum(w.conj() * e, axis=1)) ** 2
    hfss *= np.sum(hfss[TRAIN] * DATA[TRAIN]) / np.sum(hfss[TRAIN] ** 2)
    y = harmonics(th, ph, LMAX)
    return np.concatenate([y * e[:, c, None] for c in range(3)], axis=1), hfss


def best_offset(az_deg, offsets=np.arange(-45, 45.01, 0.5)):
    scores = [frac_rms(pointing_model(az_deg + off, EL)[1], TRAIN) for off in offsets]
    return float(offsets[int(np.argmin(scores))])


BETA_OFFSET = best_offset(AZ)
CMD_OFFSET = best_offset(MOTOR_AZ)
AZ_BETA = wrap(AZ + BETA_OFFSET)
AZ_CMD = wrap(MOTOR_AZ + CMD_OFFSET)
print('HFSS-selected offsets: point_table_v1-beta az %+.1f deg; commanded az = motor az %+.1f deg' % (BETA_OFFSET, CMD_OFFSET))
print('|commanded - point_table_v1-beta az|: before 20:41 median %.1f deg; after 20:54 median %.1f deg'
      % (np.median(abs(wrap(AZ_CMD - AZ_BETA))[window('20:26', '20:41')]),
         np.median(abs(wrap(AZ_CMD - AZ_BETA))[window('20:54', '21:30')])))

DESIGN_BETA, HFSS_BETA = pointing_model(AZ_BETA, EL)
DESIGN_CMD, HFSS_CMD = pointing_model(AZ_CMD, EL)
if BETA_OFFSET == 0:
    BETA, BETA_HELDOUT = SINGLE, SINGLE_HELDOUT
else:
    BETA_HELDOUT, _, _ = fit_channel(TRAIN, design=DESIGN_BETA)
    BETA, _, _ = fit_channel(FIT_OK, design=DESIGN_BETA)
CMD_HELDOUT, _, cmd_train_opt = fit_channel(TRAIN, design=DESIGN_CMD)
CMD, cmd_coeff, cmd_all_opt = fit_channel(FIT_OK, design=DESIGN_CMD)
print('commanded-az fits: train %d iterations, all-sample %d iterations' % (cmd_train_opt.nit, cmd_all_opt.nit))

regions = {'validation (held out)': FIT_OK & (SPLIT == 1), 'test (held out)': FIT_OK & (SPLIT == 2),
           'all samples': FIT_OK, 'before 20:41': FIT_OK & window('20:26', '20:41'),
           'slip 20:41-20:54': FIT_OK & window('20:41', '20:54'), 'after 20:54': FIT_OK & window('20:54', '21:30')}
rows = []
for name, model, held in [('HFSS, point_table_v1-beta az', HFSS_BETA, HFSS_BETA),
                          ('HFSS, commanded az', HFSS_CMD, HFSS_CMD),
                          ('single-channel, point_table_v1-beta az', BETA, BETA_HELDOUT),
                          ('single-channel, commanded az', CMD, CMD_HELDOUT)]:
    rows.append({'model': name, **{key: frac_rms(held if 'held out' in key else model, use)
                                   for key, use in regions.items()}})
display(pd.DataFrame(rows).set_index('model').round(4))

edges = pd.date_range('2026-07-17 20:26', '2026-07-17 21:30', freq='5min', tz='UTC')
display(pd.DataFrame([{'start': a.strftime('%H:%M'),
                       'HFSS, v1-beta': frac_rms(HFSS_BETA, FIT_OK & (tt >= a) & (tt < b)),
                       'HFSS, commanded': frac_rms(HFSS_CMD, FIT_OK & (tt >= a) & (tt < b)),
                       'single-channel, v1-beta': frac_rms(BETA, FIT_OK & (tt >= a) & (tt < b)),
                       'single-channel, commanded': frac_rms(CMD, FIT_OK & (tt >= a) & (tt < b))}
                      for a, b in zip(edges[:-1], edges[1:])]).set_index('start').round(3))

# Model-free: fractional change between the elevation profiles of consecutive sweeps.
grid = np.linspace(-170, 170, 69)
profiles = []
for k in [k for k in np.unique(SWEEP) if np.sum((SWEEP == k) & FIT_OK) > 60]:
    u = (SWEEP == k) & FIT_OK
    order = np.argsort(EL[u])
    profiles.append((np.interp(grid, EL[u][order], DATA[u][order], left=np.nan, right=np.nan),
                     np.median(AZ[u]), np.median(AZ_CMD[u]), np.median(T[u])))
pairs = []
for (p1, a1, c1, t_1), (p2, a2, c2, _) in zip(profiles[:-1], profiles[1:]):
    m = np.isfinite(p1) & np.isfinite(p2)
    pairs.append(dict(start=pd.to_datetime(t_1, unit='s', utc=True).strftime('%H:%M'),
                      pot_step=float(wrap(a2 - a1)), commanded_step=float(wrap(c2 - c1)),
                      profile_change=np.linalg.norm(p2[m] - p1[m]) / np.linalg.norm(0.5 * (p1[m] + p2[m]))))
pairs = pd.DataFrame(pairs)
stall, normal = pairs[pairs.pot_step.abs() < 2], pairs[(pairs.pot_step.abs() > 4) & (pairs.pot_step.abs() < 6)]
print('consecutive-sweep profile change: pot step < 2 deg (pot "stall"): median %.3f over %d pairs (%s); '
      'pot step 4-6 deg: median %.3f over %d pairs'
      % (stall.profile_change.median(), len(stall), ', '.join(stall.start), normal.profile_change.median(), len(normal)))

In [ ]:
plot_maps(HFSS_CMD, 'HFSS, commanded az', az=AZ_CMD)
plot_time(HFSS_CMD, 'HFSS, commanded az')

In [ ]:
plot_maps(CMD, 'single-channel ℓmax=8, commanded az', az=AZ_CMD)
plot_time(CMD, 'single-channel ℓmax=8, commanded az')

## Findings

- **The azimuth slip in `pointing_table` is the potentiometer, not the platform.** With commanded azimuth (motor steps plus one constant offset) and IMU elevation, every test improves. The strongest is the *unfitted* HFSS beam with one gain: held-out RMS drops from 0.292 to 0.132 (validation) and from 0.197 to 0.071 (test). In the slip window 20:41–20:54 it drops from 0.276 to 0.056, and before 20:41 from 0.325 to 0.077. The model-free check agrees: where the pot says the platform stalled (steps < 2°), consecutive sweeps change *more* (median 15%) than across normal 4–6° pot steps (10.5%). So the platform kept moving. `pointing_table` az is wrong by about 27° after 20:41 relative to before.
- **The 20:37–20:47 undershoot of peaks and overshoot of back lobes was this pointing error.** With commanded az, the single-channel residual there is 0.026–0.034 per 5 minutes, against 0.09–0.19 with pot az.
- **With correct pointing, HFSS is already close to the data.** The unfitted HFSS beam (test 0.071) matches the free single-channel fit (test 0.076), and beats it on validation (0.132 vs 0.159). The ℓmax = 8 free fit adds little and slightly overfits.
- **The earlier pot-az results are superseded.** That includes the finding that removing frequency bridging captures missed smooth structure: much of that "structure" was the fit absorbing pointing error. The v0003 PCA reference and the HFSS rows at the top of this notebook still use pot az.
- **What remains:** 21:11–21:20 at about 0.2 RMS for every model, which contains the whole-band dropout at 21:18 and isolated spikes of up to 4.3M counts. There is also mild large-scale structure in the HFSS residual map.
- **Caveats.**
  - The commanded offset (motor az − 9.0°) is chosen by HFSS on the training stripes. Azimuth zero is not absolute, and AZ0 = 177° was tuned against pot az.
  - The geometry (transmitter offset and polarization) was also fitted under pot az and has not been refit.
  - The single-channel fits stop at the 5,000-iteration cap. With one arm, the cross-polar component is unconstrained, so the free fit is a co-polar diagnostic only.
  - `motor_el_deg` is not usable for commanded el loops; IMU el is used throughout.

**Decision requested:**
1. Should `point_table` be rebuilt on commanded azimuth (motor steps plus offset, IMU el) and propagated to the pointing archive? That is a correction to `pointing_table`'s slip handling, so it belongs with its owner.
2. Should the geometry (az zero, transmitter offset, polarization) and the multi-frequency beam fits (v0003–v0005) be redone on commanded pointing?

STOPPED AT REVIEW GATE — awaiting Aaron's approval.